**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · AUDITORIA DO ESTUDO GLOBAL

# Guardas da seleção com treino completo

O [walkthrough](srag_course_study_walkthrough.ipynb) explica a comparação;
este caderno confere a integridade do protocolo, as listas de candidatos,
a separação do treino e a identificação do preditor escolhido.

[README do módulo](../README.md) · [protocolo](../COURSE_STUDY.md)

---

**Registro histórico interrompido:** a execução com treino completo foi encerrada
antes da seleção, por decisão de manter a amostra no novo estudo. Resultados
parciais não definem um vencedor e não substituem o preditor da aula.


In [1]:
%pip install -q numpy pandas pyarrow scikit-learn xgboost imbalanced-learn joblib matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import pathlib
import sys
import tempfile
import numpy as np
import pandas as pd
RANDOM_STATE = 42
REPO = pathlib.Path(os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    raise RuntimeError("Abra este caderno em um checkout do repositório.")
sys.path.insert(0, str(REPO / "tools"))
WORK = pathlib.Path(tempfile.mkdtemp(prefix="srag-full-study-record-"))
os.chdir(WORK)
import sapians as SP
SP.aplicar()
import srag_51_course_study as C
import srag_52_course_balance as B
RECORD = REPO / "modules/00-dataset/gold/course_study_v2.json"
r = json.loads(RECORD.read_text())
protocol = r["protocol"]
print("Versão:", protocol["version"])
print("Critério principal: menor log loss na validação temporal.")
print("Estado da execução:", r["status"])


Versão: course-full-logloss-v2
Critério principal: menor log loss na validação temporal.
Estado da execução: stopped


## §1 — Protocolo e orçamento congelados

A integridade é verificada antes de interpretar qualquer placar. As listas
correspondentes de hiperparâmetros devem ser iguais entre as estratégias.


In [3]:
payload = {k: v for k, v in protocol.items() if k != "protocol_sha256"}
assert C.object_hash(payload) == protocol["protocol_sha256"]
assert len({x["candidate_id"] for x in protocol["candidates"]}) == len(protocol["candidates"])
for family in C.BALANCED_FAMILIES:
    plain = [x["params"] for x in protocol["candidates"] if x["family"] == family and x["strategy"] == "plain"]
    for strategy in C.STRATEGIES:
        other = [x["params"] for x in protocol["candidates"] if x["family"] == family and x["strategy"] == strategy]
        assert plain == other
print("Hash do protocolo confere:", True)
print("Listas correspondentes de configurações iguais:", True)
print("Candidatos:", SP.pt_int(len(protocol["candidates"])))
print("Treino completo:", SP.pt_int(protocol["splits"]["train"]["n"]))

Hash do protocolo confere: True
Listas correspondentes de configurações iguais: True
Candidatos: 184
Treino completo: 1.242.680


## §2 — Receita e ausência de aprendizado na avaliação

As escolhas não implementadas da Gold precisam ser recusadas antes de ler os
dados. O vocabulário do pré-processamento é ajustado no treino, e um pipeline
salvo precisa produzir a mesma previsão depois de ser carregado novamente.
O recorte abaixo serve apenas para testar a integração; não estima desempenho.


In [4]:
import dataclasses
import warnings
import tempfile
import joblib
import srag_40_gold as G
import srag_60_model as M
raw = json.loads((REPO / "modules/00-dataset/gold/counts.json").read_text())["decisoes"]
raw["inicio"] = pd.Timestamp(raw["inicio"])
decisions = G.Decisions(**raw)
for key, value in (("coorte", "todas"), ("alvo", "uti"), ("etiologia", "influenza"), ("amostra_treino", 0)):
    try:
        dataclasses.replace(decisions, **{key: value})
    except ValueError:
        pass
    else:
        raise AssertionError("Receita não implementada foi aceita")
g = M.load_gold()
train = g[g["split"].eq("train")].groupby("y_obito", group_keys=False).head(32)
holdout = g[g["split"].eq("val")].head(12).copy()
category = M.CATEGORICAS[0]
holdout[category] = holdout[category].astype(str)
holdout.loc[holdout.index[0], category] = "categoria_apenas_no_teste_de_integracao"
models = {}
expected_encoder_warnings = set()
def integration_probabilities(model, frame, family):
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        probabilities = C.predict_probabilities(model, frame, family)
    for warning in captured:
        message = str(warning.message)
        if not issubclass(warning.category, UserWarning) or not message.startswith("Found unknown categories"):
            raise RuntimeError("Aviso inesperado no ensaio: " + message)
        expected_encoder_warnings.add(message)
    return probabilities
for family, params in (("logit", {"C": 0.003}), ("xgb", {"n_estimators": 2, "max_depth": 2})):
    model = C.build_pipeline(family, params, threads=1)
    model.fit(C.model_inputs(train), train["y_obito"].to_numpy())
    p_before = integration_probabilities(model, holdout, family)
    with tempfile.TemporaryDirectory() as directory:
        path = pathlib.Path(directory) / "model.joblib"
        joblib.dump(model, path)
        p_after = integration_probabilities(joblib.load(path), holdout, family)
    np.testing.assert_array_equal(p_before, p_after)
    models[family] = model
assert "categoria_apenas_no_teste_de_integracao" not in set(models["xgb"].named_steps["prepare"].named_transformers_["categorical"].categories_[0])
try:
    C._load(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
except ValueError as error:
    assert "sample" in str(error)
else:
    raise AssertionError("A amostra histórica foi aceita como treino completo")
print("Escolhas não implementadas rejeitadas:", True)
print("Categoria exclusiva da avaliação não aprendida:", True)
print("Persistência preserva previsões:", True)
print("Amostra histórica recusada para a nova seleção:", True)
for message in sorted(expected_encoder_warnings):
    print("Aviso esperado no ensaio de categorias inéditas:", message)


Escolhas não implementadas rejeitadas: True
Categoria exclusiva da avaliação não aprendida: True
Persistência preserva previsões: True
Amostra histórica recusada para a nova seleção: True
Aviso esperado no ensaio de categorias inéditas: Found unknown categories in columns [0, 18, 31, 33] during transform. These unknown categories will be encoded as all zeros


## §3 — Escolha e avaliação pertencem ao mesmo modelo

O vencedor precisa minimizar a perda de validação entre todos os candidatos
registrados. O teste exploratório não pode modificar essa escolha. Enquanto
a execução estiver incompleta, nenhum vencedor global é anunciado.


In [5]:
selection = r.get("selection")
trials = r.get("validation_candidates", [])
if selection:
    assert len(trials) == len(protocol["candidates"])
    winner = min(trials, key=lambda x: (x["validation"]["logloss"], x["candidate_id"]))
    assert winner["candidate_id"] == selection["winner"]["candidate_id"]
    assert selection["test_used_for_selection"] is False
    assert selection["protocol_sha256"] == protocol["protocol_sha256"]
    assert C.object_hash({k: v for k, v in selection.items() if k != "selection_sha256"}) == selection["selection_sha256"]
    print("Escolha corresponde ao mínimo da perda de validação:", True)
    if r.get("evaluation"):
        assert r["evaluation"]["selection_sha256"] == selection["selection_sha256"]
        assert r["evaluation"]["selection_changed"] is False
        print("Teste associado ao mesmo modelo e sem mudança da seleção:", True)
else:
    print("Execução incompleta; não existe escolha global para certificar.")

Execução incompleta; não existe escolha global para certificar.


A exportação também precisa detectar um placar adulterado. O ensaio abaixo usa
arquivos inteiramente artificiais em diretório temporário: confere uma exportação
válida e exige a rejeição de uma métrica de teste alterada, de uma seleção de
outro protocolo e de previsões modificadas. Não estima desempenho do modelo.


In [6]:
import copy
import srag_64_course_evidence as evidence_export

with tempfile.TemporaryDirectory() as temporary:
    directory = pathlib.Path(temporary).resolve()
    spec = {"candidate_id": "integrity-fixture", "family": "dummy", "strategy": "plain", "params": {}}
    fixture_protocol = {"candidates": [spec], "splits": {"test": {"n": 4}}}
    fixture_protocol["protocol_sha256"] = C.object_hash(fixture_protocol)
    C.write_json(directory / "protocol.json", fixture_protocol)
    y_fixture = np.array([0, 1, 0, 1], dtype=np.int8)
    p_fixture = np.array([0.2, 0.7, 0.1, 0.8])
    trial_directory = directory / "trials" / spec["candidate_id"]
    trial_directory.mkdir(parents=True)
    np.savez_compressed(trial_directory / "validation-predictions.npz", y=y_fixture, p=p_fixture)
    fitted_path = directory / "winner.joblib"
    fitted_path.write_bytes(b"artificial artifact for integrity checks only")
    canonical = {"path": fitted_path.name, "sha256": C.file_hash(fitted_path)}
    trial = {
        **spec, "status": "complete", "protocol_sha256": fixture_protocol["protocol_sha256"],
        "validation": C.compute_metrics(y_fixture, p_fixture),
        "validation_predictions": {
            "path": str((trial_directory / "validation-predictions.npz").relative_to(directory)),
            "sha256": C.file_hash(trial_directory / "validation-predictions.npz"),
        },
    }
    C.write_json(trial_directory / "result.json", trial)
    fixture_selection = {
        "protocol_sha256": fixture_protocol["protocol_sha256"],
        "winner": trial, "canonical_model": canonical,
    }
    fixture_selection["selection_sha256"] = C.object_hash(fixture_selection)
    C.write_json(directory / "selection.json", fixture_selection)
    np.savez_compressed(directory / "test-predictions.npz", y=y_fixture, p=p_fixture)
    fixture_evaluation = {
        "selection_sha256": fixture_selection["selection_sha256"],
        "protocol_sha256": fixture_protocol["protocol_sha256"],
        "canonical_model": canonical, "selection_changed": False,
        "test_predictions_sha256": C.file_hash(directory / "test-predictions.npz"),
        "test": C.compute_metrics(y_fixture, p_fixture),
    }
    C.write_json(directory / "test-evaluation.json", fixture_evaluation)
    assert evidence_export.project_study(directory)["status"] == "complete"

    def must_reject():
        try:
            evidence_export.project_study(directory)
        except ValueError:
            return
        raise AssertionError("A exportação aceitou evidência adulterada")

    changed = copy.deepcopy(fixture_evaluation)
    changed["test"]["logloss"] += 0.01
    C.write_json(directory / "test-evaluation.json", changed)
    must_reject()
    C.write_json(directory / "test-evaluation.json", fixture_evaluation)
    changed = copy.deepcopy(fixture_selection)
    changed["protocol_sha256"] = "another-protocol"
    changed.pop("selection_sha256")
    changed["selection_sha256"] = C.object_hash(changed)
    C.write_json(directory / "selection.json", changed)
    must_reject()
    C.write_json(directory / "selection.json", fixture_selection)
    np.savez_compressed(directory / "test-predictions.npz", y=y_fixture, p=1 - p_fixture)
    must_reject()
print("Conferido: a exportação aceita o ensaio íntegro e recusa as adulterações deliberadas.")


Conferido: a exportação aceita o ensaio íntegro e recusa as adulterações deliberadas.


## §4 — Limites da auditoria

Os testes curtos verificam guardas e persistência, não desempenho. O estudo
completo só pode ser reproduzido com a Gold e as versões registradas, usando
um diretório de saída externo. O registro agregado permite ler o estudo sem
repetir a busca ao abrir a aula. O hash do modelo identifica o ajuste
congelado; desempenho semelhante não torna ajustes diferentes equivalentes.

O teste histórico já foi observado. Nenhuma guarda computacional recupera
um período independente que ainda não tenha participado do desenvolvimento.